> **图 3.1**：对于例 3.2，预测值 $\widehat{Y}_0$（蓝色实线）和 $\widehat{Y}_0^\mathrm{L}$（红色实线）关于 $y_1\in(0,1)$

In [21]:
if(!dir.exists("data"))dir.create("data")
if(!file.exists("data/3.1-plot.RData")){
  y1=seq(0,1,length.out=200)
  y.ce=y1^2/2
  y.blup=-1/12+0.5*y1
  save(y1,y.ce,y.blup,file="data/3.1-plot.RData")
}
load("data/3.1-plot.RData")
library(svglite)
svglite("../markdown/figures/3.1.svg",width=4,height=4)
plot(y1,y.ce,type="l",xlab="y1",ylab="y0 预测量",xlim=c(0,1))
lines(y1,y.blup,lty=2)
dev.off()


agg_record_68ac242a68ce 
                      2

> **图 3.2**：示例 1.1 所用 $40$ 个输入点的散点图矩阵

In [22]:
load("data/1.1-plot.RData")
library(svglite)
svglite("../markdown/figures/3.2.svg",width=8,height=8)
pairs(df[,c("A","H","F","LC")],pch=16,labels=c("房间面积","房间高度","火焰高度","损失比例"))
dev.off()

agg_record_68ac13e5295e 
                      2

> **图 3.3**：利用例 1.1 的数据绘制火灾蔓延至火源上方 $5$ 英尺所需时间分别与各输入参数的散点图：（1）热损失系数（2）火源距地面高度（3）房间高度（4）房间面积

In [23]:
load("data/1.1-plot.RData")
library(svglite)
svglite("../markdown/figures/3.3.svg",width=8,height=8)
par(mfrow=c(2,2))
plot(df$LC,df$y,pch=16,xlab="热损失系数",ylab="到达5英尺所需时间（秒）")
plot(df$F,df$y,pch=16,xlab="火源距地面高度（英尺）",ylab="到达5英尺所需时间（秒）")
plot(df$H,df$y,pch=16,xlab="房间高度（英尺）",ylab="到达5英尺所需时间（秒）")
plot(df$A,df$y,pch=16,xlab="房间面积（平方英尺）",ylab="到达5英尺所需时间（秒）")
par(mfrow=c(1,1))
dev.off()


agg_record_68ac11292863 
                      2

> **表 3.2**：式（3.3.12）中 $\xi_j$（$j=1,\dots,4$）的限制极大似然估计值，以及各输入对应的粗略灵敏度指标 $\exp\left(-\xi_jr_j^2\right)$（数值越小代表影响程度越高）

In [24]:
if(!dir.exists("data"))dir.create("data")
if(!file.exists("data/3.2-fit.RData")){
  load("data/1.1-plot.RData")
  X=as.matrix(df[,c("LC","F","H","A")]);y=df$y;n=nrow(X);one=rep(1,n)
  nugget=1e-4
  neg_REML=function(lnxi){
    xi=exp(lnxi)
    D2=matrix(0,n,n)
    for(j in 1:4)D2=D2+outer(X[,j],X[,j],function(a,b)xi[j]*(a-b)^2)
    Rm=exp(-D2)+diag(n)*nugget
    C=chol(Rm);Rinv=chol2inv(C)
    beta=as.numeric((t(one)%*%Rinv%*%y)/(t(one)%*%Rinv%*%one))
    resid=y-one*beta
    s2=as.numeric(t(resid)%*%Rinv%*%resid)/(n-1)
    -(-sum(log(diag(C)))-0.5*(n-1)*log(s2)-0.5*log(as.numeric(t(one)%*%Rinv%*%one)))
  }
  fit=optim(log(c(1,0.05,0.01,0.0001)),neg_REML,method="L-BFGS-B",lower=log(1e-7),upper=log(100),control=list(maxit=5000,factr=1e-12))
  xi=exp(fit$par)
  D2=matrix(0,n,n)
  for(j in 1:4)D2=D2+outer(X[,j],X[,j],function(a,b)xi[j]*(a-b)^2)
  Rm=exp(-D2)+diag(n)*nugget
  Rinv=chol2inv(chol(Rm))
  beta0=as.numeric((t(one)%*%Rinv%*%y)/(t(one)%*%Rinv%*%one))
  resid=y-one*beta0
  s2=as.numeric(t(resid)%*%Rinv%*%resid)/(n-1)
  save(xi,beta0,s2,file="data/3.2-fit.RData")
}
load("data/3.2-fit.RData")
load("data/1.1-plot.RData")
cat("hat(beta0):",round(beta0,4),"\that(sigma2):",round(s2,4),"\n\n")
cat("表 3.2:\n")
X=as.matrix(df[,c("LC","F","H","A")])
r2=apply(X,2,function(v)diff(range(v)))^2
cn=c(LC="热损失系数",F="火源高度",H="房间高度",A="房间面积")
tab=cbind(xi=xi,xir2=xi*r2,expxi=exp(-xi*r2))
rownames(tab)=cn[names(r2)]
print(round(tab,4))

hat(beta0): 12.454 	hat(sigma2): 167.663 

表 3.2:
               xi   xir2  expxi
热损失系数 0.1905 0.0151 0.9850
火源高度   0.0485 0.1763 0.8384
房间高度   0.0350 0.5092 0.6010
房间面积   0.0000 0.1464 0.8638


> **图 3.4**：示例 1.1 中使用的 $320$ 个等距网格点，火源上方 $5$ 英尺处真实到达时间与预测到达时间的散点图

In [26]:
if(!dir.exists("data"))dir.create("data")
have34=file.exists("data/3.4-plot.RData")&&{
  e0=new.env();load("data/3.4-plot.RData",envir=e0)
  all(c("cv.pred","rmspe.cv","rmspe.te")%in%ls(e0))
}
if(!have34){
  load("data/1.1-plot.RData")
  load("data/3.2-fit.RData")
  CT=1;CL=1;G=32.2;CP=0.24;PA=530.0;DA=0.075;LR=0.35;Q0=0.1;targetZ=5
  QA_peak=1000;tpk=180;TMAX=1200
  QAt=function(t)if(t<=tpk)QA_peak*t/tpk else QA_peak
  QTfun=function(t)QAt(t)/Q0
  X=as.matrix(df[,c("LC","F","H","A")]);y=df$y;n=nrow(X);one=rep(1,n)
  nugget=1e-4
  D2=matrix(0,n,n)
  for(j in 1:4)D2=D2+outer(X[,j],X[,j],function(a,b)xi[j]*(a-b)^2)
  Rm=exp(-D2)+diag(n)*nugget
  Rinv=chol2inv(chol(Rm))
  krig=function(x0){
    r=exp(-rowSums(sweep(X,2,x0)^2*rep(xi,each=n)))
    beta0+as.numeric(t(r)%*%Rinv%*%(y-one*beta0))
  }
  lc=seq(0.6,0.9,length.out=4);h=seq(8,12,length.out=4);f=seq(1,3,length.out=4);a=seq(81,256,length.out=5)
  grid=expand.grid(LC=lc,H=h,F=f,A=a)
  ytrue=apply(grid,1,function(r)asetb_orig(A=r["A"],H=r["H"],F=r["F"],LC=r["LC"]))
  ypred=apply(grid,1,function(r)krig(as.numeric(r[c("LC","F","H","A")])))
  rmspe.te=sqrt(mean((ytrue-ypred)^2))
  neg_REML=function(lnxi,Xs,ys){
    xi=exp(lnxi);ns=nrow(Xs);one=rep(1,ns)
    D2=matrix(0,ns,ns)
    for(j in 1:4)D2=D2+outer(Xs[,j],Xs[,j],function(a,b)xi[j]*(a-b)^2)
    Rm=exp(-D2)+diag(ns)*nugget
    C=chol(Rm);Rinv=chol2inv(C)
    beta=as.numeric((t(one)%*%Rinv%*%ys)/(t(one)%*%Rinv%*%one))
    resid=ys-one*beta
    s2=as.numeric(t(resid)%*%Rinv%*%resid)/(ns-1)
    -(-sum(log(diag(C)))-0.5*(ns-1)*log(s2)-0.5*log(as.numeric(t(one)%*%Rinv%*%one)))
  }
  cv.pred=numeric(n)
  for(i in 1:n){
    sel=setdiff(1:n,i);Xs=X[sel,,drop=FALSE];ys=y[sel];ns=n-1
    fit=optim(log(xi),neg_REML,Xs=Xs,ys=ys,method="L-BFGS-B",lower=log(1e-7),upper=log(100),control=list(maxit=5000,factr=1e-12))
    xii=exp(fit$par);one=rep(1,ns)
    D2=matrix(0,ns,ns)
    for(j in 1:4)D2=D2+outer(Xs[,j],Xs[,j],function(a,b)xii[j]*(a-b)^2)
    Rm=exp(-D2)+diag(ns)*nugget
    Rinv=chol2inv(chol(Rm))
    beta0i=as.numeric((t(one)%*%Rinv%*%ys)/(t(one)%*%Rinv%*%one))
    r=exp(-rowSums(sweep(Xs,2,X[i,])^2*rep(xii,each=ns)))
    cv.pred[i]=beta0i+as.numeric(t(r)%*%Rinv%*%(ys-one*beta0i))
  }
  rmspe.cv=sqrt(mean((y-cv.pred)^2))
  save(xi,beta0,s2,ytrue,ypred,cv.pred,rmspe.cv,rmspe.te,file="data/3.4-plot.RData")
}
load("data/3.4-plot.RData")
cat("Cross-Val RMSPE =",round(rmspe.cv,4),"\n")
cat("Test 网格 RMSPE =",round(rmspe.te,4),"\n")
library(svglite)
svglite("../markdown/figures/3.4.svg",width=4,height=4)
plot(ytrue,ypred,pch=16,xlab="真实到达时间（秒）",ylab="预测到达时间（秒）")
abline(0,1,lty=2)
dev.off()

Cross-Val RMSPE = 0.2377 
Test 网格 RMSPE = 1.347 


agg_record_68ac1e9538c3 
                      2